In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/config.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/X_train_pre.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/camembert_training_history.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/images_analysis.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/lang.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/translation.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_train_clean.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/label_mapping.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_train.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_val.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/tokenizer.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/tokenizer_config.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/vgg16_best.pth
/kaggle/input/datasets/tarikouhsine/raku

KeyboardInterrupt: 

## CELLULE 1 — Environnement

In [6]:
import os
from pathlib import Path

if os.path.exists('/kaggle'):
    INPUT_DIR  = Path('/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    BASE_DIR   = Path(os.getcwd()).parent
    INPUT_DIR  = BASE_DIR / 'data' / 'processed'
    OUTPUT_DIR = BASE_DIR / 'data' / 'processed'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input  : {INPUT_DIR}")
print(f"Output : {OUTPUT_DIR}")

Input  : /kaggle/input/datasets/tarikouhsine/rakuten-preprocessed
Output : /kaggle/working


## CELLULE 2 — Imports

In [8]:
!pip install googletrans==4.0.0-rc1 -q

import pandas as pd
from googletrans import Translator, LANGUAGES
import json
import os
from requests.exceptions import ReadTimeout

## CELLULE 3 — Chargement

In [9]:
df = pd.read_csv(INPUT_DIR / 'X_train_pre.csv', index_col=0)
print(df.head())

                                         designation  \
0  Olivia: Personalisiertes Notizbuch / 150 Seite...   
1  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...   
2  Grand Stylet Ergonomique Bleu Gamepad Nintendo...   
3  Peluche Donald - Europe - Disneyland 2000 (Mar...   
4                               La Guerre Des Tuques   

                                         description  \
0                                                NaN   
1                                                NaN   
2  PILOT STYLE Touch Pen de marque Speedlink est ...   
3                                                NaN   
4  Luc a des idées de grandeur. Il veut organiser...   

                                                text   productid     imageid  \
0  Olivia: Personalisiertes Notizbuch / 150 Seite...  3804725264  1263597046   
1  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...   436067568  1008141237   
2  Grand Stylet Ergonomique Bleu Gamepad Nintendo...   201115110   938777978   
3  Pel

## CELLULE 4 — Fonction de traduction

In [12]:
# Initialiser le traducteur
translator = Translator()

def translate_text(df, start_index=0,
                   save_path=str(OUTPUT_DIR / 'translation.json')):
    """
    Traduit la colonne texte d'un DataFrame vers le français
    et sauvegarde le résultat dans save_path au fur et à mesure.
    
    df          : DataFrame avec colonnes 'text' et 'lang'
    start_index : permet de reprendre après une erreur
    save_path   : chemin de sauvegarde du fichier de traduction
    
    Attention : environ 750 lignes / 10 minutes
                84000 lignes → environ 20h
    """
    translations = {}
    if os.path.exists(save_path):
        with open(save_path, 'r') as file:
            translations = json.load(file)

    if start_index > len(df):
        print("Job terminé : Aucune ligne restante à traduire.")
        return

    for i, row in df.iloc[start_index:].iterrows():
        try:
            if row['lang'] in LANGUAGES and row['lang'] != 'fr':
                texte    = row['text']
                src_lang = row['lang']
                if len(texte) > 4000:
                    parts         = [texte[j:j+4000]
                                     for j in range(0, len(texte), 4000)]
                    translated_text = ' '.join([
                        translator.translate(part, src=src_lang, dest='fr').text
                        for part in parts
                    ])
                else:
                    translated_text = translator.translate(
                        texte, src=src_lang, dest='fr').text
            else:
                translated_text = row['text']

            translations[i] = translated_text

            with open(save_path, 'w') as file:
                json.dump(translations, file)

        except ReadTimeout:
            print(f"ReadTimeout à l'index {i}. Tentative de redémarrage.")
            translate_text(df, start_index=i, save_path=save_path)
            break

        except AttributeError as e:
            print(f"Erreur à l'index {i}: {e}")
            break

## CELLULE 5 — Test de la fonction

In [13]:
test = pd.DataFrame({
    'text' : ['Hello this is a test', 'Guten Tag', 'Bonjour le monde'],
    'lang' : ['en', 'de', 'fr']
}, index=[0, 1, 2])

test_path = str(OUTPUT_DIR / 'translation_test.json')

try:
    translations_dict = pd.read_json(test_path, orient='index')
    translations_df   = translations_dict.rename(columns={0: 'text_fr'})
    translate_text(test, start_index=translations_df.value_counts().sum(),
                   save_path=test_path)
except (FileNotFoundError, ValueError):
    translate_text(test, save_path=test_path)

translations_dict = pd.read_json(test_path, orient='index')
translations_df   = translations_dict.rename(columns={0: 'text_fr'})
print(translations_df)
translations_df.value_counts().sum()

                  text_fr
0  Bonjour, c'est un test
1           Bonne journée
2        Bonjour le monde


np.int64(3)

## CELLULE 6 — Traduction du dataset complet 
Attention : environ 750 lignes / 10 minutes _ 84000 lignes → environ 20h

In [ ]:
# Environ 750 lignes / 10 minutes — 84000 lignes → environ 20h
# Le fichier translation.json produit est disponible dans le dataset Kaggle

save_path = str(OUTPUT_DIR / 'translation.json')

try:
    translations_dict = pd.read_json(save_path, orient='index')
    translations_df   = translations_dict.rename(columns={0: 'text_fr'})
    translate_text(df, start_index=translations_df.value_counts().sum(),
                   save_path=save_path)
except (FileNotFoundError, ValueError):
    translate_text(df, save_path=save_path)

## CELLULE 7 — Vérification

In [15]:
translations_dict = pd.read_json(str(OUTPUT_DIR / 'translation_test.json'),
                                 orient='index')
translations_df   = translations_dict.rename(columns={0: 'text_fr'})

print(translations_df.tail(10))
print(f"\nNombre de traductions : {translations_df.value_counts().sum()}")

                  text_fr
0  Bonjour, c'est un test
1           Bonne journée
2        Bonjour le monde

Nombre de traductions : 3
